In [1]:
!pip install numpy pandas scikit-learn tensorflow joblib

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
import joblib

# 1. Membaca dataset yang ada di folder laptopmu
# Jalankan ini jika file csv berada satu folder dengan file notebook ini
df = pd.read_csv('dataset/diabetes_binary_health_indicators_BRFSS2015.csv')

# Menampilkan 5 data teratas untuk memastikan data terbaca
df.head()

,Diabetes_binary,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,...,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income
0,0.0,1.0,1.0,1.0,40.0,1.0,0.0,0.0,0.0,0.0,...,1.0,0.0,5.0,18.0,15.0,1.0,0.0,9.0,4.0,3.0
1,0.0,0.0,0.0,0.0,25.0,1.0,0.0,0.0,1.0,0.0,...,0.0,1.0,3.0,0.0,0.0,0.0,0.0,7.0,6.0,1.0
2,0.0,1.0,1.0,1.0,28.0,0.0,0.0,0.0,0.0,1.0,...,1.0,1.0,5.0,30.0,30.0,1.0,0.0,9.0,4.0,8.0
3,0.0,1.0,0.0,1.0,27.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,0.0,0.0,0.0,0.0,11.0,3.0,6.0
4,0.0,1.0,1.0,1.0,24.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,3.0,0.0,0.0,0.0,11.0,5.0,4.0


In [3]:
# Masukkan nama kolom yang sudah disesuaikan dengan dataset aslimu ('NoDocbcCost')
features = [
    'HighBP', 'HeartDiseaseorAttack', 'Smoker', 'HvyAlcoholConsump', 
    'PhysActivity', 'Fruits', 'Veggies', 'NoDocbcCost', 'DiffWalk', 
    'MentHlth', 'PhysHlth', 'Age', 'Sex', 'BMI'
]

# Ambil fitur dan target
X = df[features]
y = df['Diabetes_binary']

# Membagi data menjadi Data Training (80%) dan Data Testing (20%)
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print("Fitur yang digunakan:", list(X.columns))
print("Jumlah fitur wajib:", X.shape[1])
print("Data Training:", X_train.shape)
print("Data Testing:", X_test.shape)

Fitur yang digunakan: ['HighBP', 'HeartDiseaseorAttack', 'Smoker', 'HvyAlcoholConsump', 'PhysActivity', 'Fruits', 'Veggies', 'NoDocbcCost', 'DiffWalk', 'MentHlth', 'PhysHlth', 'Age', 'Sex', 'BMI']
Jumlah fitur wajib: 14
Data Training: (202944, 14)
Data Testing: (50736, 14)


In [4]:
from sklearn.preprocessing import StandardScaler
import joblib

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Simpan scaler agar bisa dipakai di FastAPI nanti
joblib.dump(scaler, 'scaler_treehealthy.pkl')
print("Scaler 14 fitur berhasil disimpan!")

Scaler 14 fitur berhasil disimpan!


In [6]:
from xgboost import XGBClassifier
import joblib

print("Memulai pelatihan XGBoost...")

# Inisialisasi XGBoost dengan parameter dasar
xgb_model = XGBClassifier(
    n_estimators=100, 
    max_depth=6, 
    learning_rate=0.1, 
    random_state=42, 
    use_label_encoder=False, 
    eval_metric='logloss'
)

# Proses Training
xgb_model.fit(X_train_scaled, y_train)

# Evaluasi
xgb_preds = xgb_model.predict(X_test_scaled)
print(f"Akurasi XGBoost: {accuracy_score(y_test, xgb_preds) * 100:.2f}%")

# Simpan model untuk FastAPI nanti
joblib.dump(xgb_model, 'model_xgb_treehealthy.pkl')

Memulai pelatihan XGBoost...


c:\Users\Ezra\AppData\Local\Programs\Python\Python313\Lib\site-packages\xgboost\training.py:200: UserWarning: [14:38:40] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:793: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Akurasi XGBoost: 86.25%


['model_xgb_treehealthy.pkl']

In [7]:
print("Menyusun Arsitektur Deep Learning MLP...")

# Bangun struktur jaringan saraf tiruan berlapis
mlp_model = Sequential([
    # Layer Input & Hidden Layer 1 (64 Neuron)
    Dense(64, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    Dropout(0.2), # Mencegah ketergantungan berlebih antar neuron
    
    # Hidden Layer 2 (32 Neuron)
    Dense(32, activation='relu'),
    Dropout(0.2),
    
    # Output Layer (1 Neuron dengan Sigmoid untuk klasifikasi biner 0 atau 1)
    Dense(1, activation='sigmoid')
])

# Kompilasi model dengan optimizer Adam dan fungsi loss Binary Crossentropy
mlp_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Amankan proses training agar berhenti jika tidak ada peningkatan lagi
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

print("Memulai pelatihan Deep Learning (ini mungkin memakan waktu beberapa menit)...")
# Mulai proses training sebanyak 30 putaran (epochs)
history = mlp_model.fit(
    X_train_scaled, y_train,
    epochs=30,
    batch_size=64,
    validation_split=0.2, # Menyisihkan 20% data training untuk validasi tiap putaran
    callbacks=[early_stop],
    verbose=1
)

Menyusun Arsitektur Deep Learning MLP...
Memulai pelatihan Deep Learning (ini mungkin memakan waktu beberapa menit)...
Epoch 1/30


c:\Users\Ezra\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


2537/2537 ━━━━━━━━━━━━━━━━━━━━ 4s 1ms/step - accuracy: 0.8601 - loss: 0.3421 - val_accuracy: 0.8651 - val_loss: 0.3270
Epoch 2/30
2537/2537 ━━━━━━━━━━━━━━━━━━━━ 5s 1ms/step - accuracy: 0.8621 - loss: 0.3344 - val_accuracy: 0.8655 - val_loss: 0.3261
Epoch 3/30
2537/2537 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step - accuracy: 0.8633 - loss: 0.3329 - val_accuracy: 0.8657 - val_loss: 0.3254
Epoch 4/30
2537/2537 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step - accuracy: 0.8629 - loss: 0.3320 - val_accuracy: 0.8659 - val_loss: 0.3266
Epoch 5/30
2537/2537 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step - accuracy: 0.8629 - loss: 0.3322 - val_accuracy: 0.8659 - val_loss: 0.3261
Epoch 6/30
2537/2537 ━━━━━━━━━━━━━━━━━━━━ 5s 1ms/step - accuracy: 0.8631 - loss: 0.3317 - val_accuracy: 0.8660 - val_loss: 0.3258
Epoch 7/30
2537/2537 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step - accuracy: 0.8630 - loss: 0.3317 - val_accuracy: 0.8662 - val_loss: 0.3269
Epoch 8/30
2537/2537 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step - accuracy: 0.8632 - loss: 0.3316 - val_accurac

In [8]:
# Evaluasi model pada data testing yang belum pernah dilihat sama sekali
loss, accuracy = mlp_model.evaluate(X_test_scaled, y_test, verbose=0)
print(f"\n=== PERFORMA DEEP LEARNING MLP ===")
print(f"Akurasi Akhir: {accuracy * 100:.2f}%")

# Simpan model Deep Learning ke dalam format file keras
mlp_model.save('model_mlp_treehealthy.h5')
print("Model Deep Learning berhasil disimpan!")


=== PERFORMA DEEP LEARNING MLP ===
Akurasi Akhir: 86.28%
Model Deep Learning berhasil disimpan!
